# Speaker Diarization Tutorial

This notebook walks through how to:
1. Download audio from YouTube
2. Transcribe speech to text with **OpenAI Whisper**
3. Run **speaker diarization** with `pyannote.audio` (who spoke when)
4. Combine both to produce a labeled transcript: `[SPEAKER_00] 12.3s: Hello...`

The pipeline is a simple introduction to this method.

## 1. Install dependencies

Run this once in your environment. The `pyannote/speaker-diarization-community-1` model requires a Hugging Face account and token.

In [ ]:
pip install pyannote.audio openai-whisper av yt-dlp scipy

You also need a Hugging Face token with access to the model.

**Local environment** — set it as an environment variable before launching Jupyter:
```bash
# Windows PowerShell
$env:HF_TOKEN = "hf_..."

# Linux/macOS
export HF_TOKEN="hf_..."
```

**Google Colab** — use the Secrets manager instead (key icon in the left sidebar):
1. Click the key icon → **Add new secret**
2. Name: `HF_TOKEN`, Value: your token
3. Toggle **Notebook access** on

Then load it in your notebook:
```python
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
```

Accept the model license at: https://huggingface.co/pyannote/speaker-diarization-community-1

For Google Colabs:

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## 2. Imports

In [ ]:
import os
import numpy as np
import torch
import av
from math import gcd
from scipy.signal import resample_poly
import whisper
from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

## 3. Download test audio

We use `yt_dlp` to download the best-quality audio stream from a YouTube video. The result is a `.webm` file (Opus-encoded audio).

The video used here is this two minute NPR video: https://www.youtube.com/watch?v=ThN7CkeEXXk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/Teaching"
VIDEO_ID  = "ThN7CkeEXXk"
webm_path = f"{DATA_DIR}/{VIDEO_ID}.webm"

if not os.path.exists(webm_path):
    import yt_dlp
    ydl_opts = {
        "format": "bestaudio",
        "outtmpl": f"{DATA_DIR}/{VIDEO_ID}.%(ext)s",
        "noplaylist": True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([f"https://www.youtube.com/watch?v={VIDEO_ID}"])
else:
    print("Audio already downloaded.")

## 4. Load audio into a tensor

`pyannote` expects audio as a dict `{"waveform": tensor, "sample_rate": int}` where `waveform` has shape `(channels, samples)`.

We use PyAV to decode the audio file directly into a NumPy array, then wrap it in a PyTorch tensor.

Note: This might take a few minutes to run!

In [ ]:
container = av.open(webm_path)
audio_stream = container.streams.audio[0]

# Decode every frame and stack them along the time axis
frames = [frame.to_ndarray() for frame in container.decode(audio_stream)]
waveform = torch.tensor(np.concatenate(frames, axis=1), dtype=torch.float32)

audio_input = {"waveform": waveform, "sample_rate": audio_stream.sample_rate}
container.close()

print(f"Waveform shape: {waveform.shape}  (channels × samples)")
print(f"Sample rate:    {audio_stream.sample_rate} Hz")
print(f"Duration:       {waveform.shape[1] / audio_stream.sample_rate:.1f} s")

## 4. Transcribe with Whisper

Whisper expects **mono, float32 audio at 16 kHz**. Our waveform may be stereo and at a different sample rate (e.g., 48 kHz), so we need to:
1. Mix channels down to mono
2. Resample to 16 kHz using `scipy.signal.resample_poly`

We use `resample_poly` (polyphase resampling). It takes an upsample factor and a downsample factor based on the GCD of the two sample rates.

Available Whisper model sizes: `tiny`, `base`, `small`, `medium`, `large`. Larger models are more accurate but slower. For Spanish transcription, `medium` or `large` is recommended.

In [ ]:
# Step 1: Convert waveform tensor to a NumPy array
audio_np = waveform.numpy()
print(f"Original shape: {audio_np.shape}  (channels × samples)")

# Step 2: Mix to mono
if audio_np.shape[0] > 1:
    audio_np = audio_np.mean(axis=0)   # average across channels
else:
    audio_np = audio_np[0]
print(f"Mono shape:     {audio_np.shape}")

# Step 3: Resample to 16 kHz if needed
src_rate = audio_stream.sample_rate
target_rate = 16000
if src_rate != target_rate:
    g = gcd(src_rate, target_rate)
    audio_np = resample_poly(audio_np, target_rate // g, src_rate // g)
    print(f"Resampled from {src_rate} Hz \u2192 {target_rate} Hz")

audio_np = audio_np.astype(np.float32)
print(f"Final shape:    {audio_np.shape}")

In [ ]:
# Load Whisper model and transcribe
# verbose=True prints each segment as it's decoded
model = whisper.load_model("medium")

result = model.transcribe(audio_np, task="transcribe", language="en", fp16=False, verbose=True)

In [ ]:
for seg in result["segments"]:
      print(
          f"{seg['start']:6.1f}s | "
          f"no_speech={seg['no_speech_prob']:.2f} | "
          f"logprob={seg['avg_logprob']:.2f} | "
          f"compression={seg['compression_ratio']:.2f} | "
          f"temp={seg['temperature']} | "
          f"{seg['text'].strip()}"
      )

## 6. Load the pyannote diarization pipeline

The pipeline is downloaded from Hugging Face on first use and cached locally. It requires `HF_TOKEN` in your environment and that you have accepted the model license.

We move the pipeline to the GPU if one is available — diarization is much faster on CUDA.

In [ ]:
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=os.environ["HF_TOKEN"]
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pipeline.to(device)
print(f"Pipeline running on: {device}")

## 7. Run speaker diarization

The pipeline returns an `Annotation` object containing time-stamped speaker turns. Each turn has a start time, end time, and a speaker label like `SPEAKER_00`, `SPEAKER_01`, etc.

The labels are arbitrary: `SPEAKER_00` is just whichever speaker the model encountered first. You'd need a separate speaker identification step to map them to real names.

Note: This may take awhile!

In [ ]:
with ProgressHook() as hook:
    output = pipeline(audio_input, hook=hook)

In [ ]:
# Print the diarization result
print("\nDiarization output:")
for turn, _, speaker in output.speaker_diarization.itertracks(yield_label=True):
    print(f"  {speaker}  {turn.start:6.1f}s → {turn.end:6.1f}s")

## 8. Assign speakers to transcript segments

Whisper returns a list of `segments`, each with `start`, `end`, and `text`. We match each segment to the diarization output by finding which speaker has the most overlap with that time window.

The `get_speaker` function iterates over all diarization turns and returns the speaker whose turn overlaps the most with the Whisper segment.

In [ ]:
def get_speaker(start, end, diarization):
    """Return the speaker with the most overlap with the time window [start, end]."""
    max_overlap = 0.0
    best_speaker = None
    best_distance = float("inf")

    for turn, _, speaker in diarization.itertracks(yield_label=True):
        overlap = min(turn.end, end) - max(turn.start, start)
        if overlap > max_overlap:
            max_overlap = overlap
            best_speaker = speaker
        distance = min(abs(turn.start - start), abs(turn.end - end))
        if overlap <= 0 and distance < best_distance:
            best_distance = distance
            if best_speaker is None:
                best_speaker = speaker

    return best_speaker or "UNKNOWN"

In [ ]:
# Print the labeled transcript
print("Labeled transcript:\n")
for seg in result["segments"]:
    speaker = get_speaker(seg["start"], seg["end"], output.speaker_diarization)
    print(f"[{speaker}] {seg['start']:6.1f}s: {seg['text'].strip()}")

## 9. Export to a DataFrame

It's useful to store the labeled transcript as a CSV for downstream analysis.

In [15]:
import pandas as pd

rows = []
for seg in result["segments"]:
    speaker = get_speaker(seg["start"], seg["end"], output.speaker_diarization)
    rows.append({
        "speaker": speaker,
        "start": seg["start"],
        "end": seg["end"],
        "text": seg["text"].strip(),
    })

df = pd.DataFrame(rows)
df.head(10)

,speaker,start,end,text
0,SPEAKER_05,0.00,4.84,What do you think about the current situation?
1,SPEAKER_05,4.84,7.34,Everything's high. Everything's high.
2,SPEAKER_05,7.34,8.84,You can barely help yourself.
3,SPEAKER_05,8.84,12.34,The market rent is high.
4,SPEAKER_05,12.34,16.26,It's almost impossible to be able to even come...
5,SPEAKER_05,16.26,18.86,let alone two times the rent and the deposit.
6,SPEAKER_06,18.86,22.46,I hope the people that typically vote Republican
7,SPEAKER_06,22.46,27.76,are thinking about what the GOP has done for them
8,SPEAKER_06,27.76,31.26,on what they were running on a couple of years...
9,SPEAKER_06,31.26,33.70,"No wars, better economy, blah, blah, blah."


In [16]:
df["speaker"].value_counts()

speaker
SPEAKER_04    9
SPEAKER_07    7
SPEAKER_05    6
SPEAKER_03    6
SPEAKER_01    6
SPEAKER_08    6
SPEAKER_06    5
SPEAKER_02    3
SPEAKER_00    3
Name: count, dtype: int64

In [17]:
out_path = f"{DATA_DIR}/{VIDEO_ID}_diarized.csv"
df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to /content/drive/MyDrive/Teaching/ThN7CkeEXXk_diarized.csv


## Notes and limitations

- **pyannote works best with 1–10 speakers.** Crowded press-conference rooms may confuse the model.
- **Whisper timestamps are approximate.** The `get_speaker` overlap heuristic handles small misalignments, but errors increase with fast speaker switches.
- **GPU strongly recommended.** On CPU, a 5-minute chunk takes roughly 5–10 minutes to diarize and transcribe.